# PAUDC Asset Pipeline — Free Kaggle GPU

Runs the project's AI asset steps on Kaggle's **free GPU tier** (~30 GPU-hrs/week).
Upload this notebook at https://www.kaggle.com/code → *New Notebook* → *File → Import Notebook*,
then in **Settings → Accelerator** pick **GPU T4 x2** (or P100).

**What it does**
1. Verifies the GPU.
2. **Texture upscale:** regenerates the game's procedural textures (facade / asphalt / ground noise —
   same algorithms as `game/3d.html`) at base resolution, then upscales them 4× with **Real-ESRGAN**
   (open source) for the engine-rung builds.
3. **Image → 3D (optional):** turns an original concept image into a textured GLB prop with an
   open-source image-to-3D model — the Hunyuan3D-class step from the roadmap.
4. Saves everything to `/kaggle/working/out/` — download and commit as ordinary files.

**Rules (from `worldbuilding/DEVELOPMENT_ROADMAP.md`):** original inputs only — our prompts, our
concept art. Never feed anyone else's IP through the models. Fictional game content only.


In [ ]:
# 1) GPU check
import torch, subprocess
print('CUDA available:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout)


In [ ]:
# 2a) Regenerate the game's procedural textures at base res (same recipes as game/3d.html)
import numpy as np, os
from PIL import Image, ImageDraw
os.makedirs('/kaggle/working/out', exist_ok=True)
rng = np.random.default_rng(7)

def save(name, arr):
    Image.fromarray(arr).save(f'/kaggle/working/out/{name}.png'); print('wrote', name)

# ground detail noise (the terrain multiply map)
g = np.full((128,128,3), 143, np.uint8)
for _ in range(2600):
    x,y = rng.integers(0,127,2); v = int(120+rng.random()*46)
    g[y:y+2, x:x+2] = (v,v,v)
save('ground_detail_base', g)

# building facade (lit/dark window grid)
im = Image.new('RGB',(128,128),(207,207,207)); d = ImageDraw.Draw(im)
for y in range(10,114,26):
    for x in range(10,114,24):
        lit = rng.random() < 0.42
        d.rectangle([x,y,x+14,y+16], fill=(255,217,138) if lit else (26,37,48))
        d.rectangle([x,y+14,x+14,y+16], fill=(60,60,60))
save('facade_base', np.asarray(im))

# asphalt + dashed centerline
r = np.zeros((64,128,3), np.uint8); r[:,:] = (35,43,51)
for _ in range(900):
    x,y = rng.integers(0,127), rng.integers(0,63); v = int(28+rng.random()*26)
    r[y:y+2, x:x+2] = (v,v,v+6)
for x in range(0,128,32): r[30:34, x:x+18] = (216,207,154)
save('asphalt_base', r)


In [ ]:
# 2b) 4x upscale with Real-ESRGAN (open source) -> engine-res textures
%pip -q install realesrgan basicsr facexlib gfpgan
import glob
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
up = RealESRGANer(scale=4, model=model,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v1.0/RealESRGAN_x4plus.pth',
    tile=0, half=torch.cuda.is_available())
for p in glob.glob('/kaggle/working/out/*_base.png'):
    img = np.asarray(Image.open(p))
    outp, _ = up.enhance(img, outscale=4)
    Image.fromarray(outp).save(p.replace('_base','_4x'))
    print('upscaled', p)


## 3) Optional: image → 3D prop (GLB)
Attach your **original** concept image as a Kaggle dataset (or upload into `/kaggle/working`),
set `CONCEPT` below, and run. Uses an open image-to-3D checkpoint via `diffusers`/vendor repo —
on the free T4 use the *mini/turbo* variants (full Hunyuan3D-2.1 PBR wants ~29 GB VRAM; the
shape-only mini variants fit in 16 GB). Output GLB drops in `/kaggle/working/out/`.
Then do the game-ready pass (decimate, LODs, collision) before engine import — see roadmap.


In [ ]:
CONCEPT = ''  # e.g. '/kaggle/input/my-concepts/mudfish_side.png' — leave empty to skip
if CONCEPT:
    %pip -q install hy3dgen trimesh
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mini')
    mesh = pipe(image=CONCEPT)[0]
    mesh.export('/kaggle/working/out/prop.glb')
    print('wrote prop.glb')
else:
    print('CONCEPT empty — skipped (texture steps above already ran)')


---
**Download** `/kaggle/working/out/` from the right-hand *Output* panel and commit the files to the
repo. The browser game stays dependency-free — these assets feed the Godot/UE5 engine rung.
